In [1]:
import os
import json
import pickle
import faiss
import numpy as np

from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PATHS = {
    "chunks": DATA_PROCESSED_DIR / "chunks.json",
    "faiss_index": DATA_PROCESSED_DIR / "faiss_index.bin",
    "bm25_index": DATA_PROCESSED_DIR / "bm25_index.pkl"
}

for name, path in PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")

with open(PATHS["chunks"], "r", encoding="utf-8") as f:
    chunks = json.load(f)

faiss_index = faiss.read_index(str(PATHS["faiss_index"]))

with open(PATHS["bm25_index"], "rb") as f:
    bm25_index = pickle.load(f)

print(f"Total Chunks in Registry : {len(chunks)}")
print(f"FAISS Index Vectors      : {faiss_index.ntotal}")
print(f"BM25 Corpus Size         : {bm25_index.corpus_size}")

assert len(chunks) == faiss_index.ntotal == bm25_index.corpus_size

print("\nAll indexes loaded and verified successfully!")

Total Chunks in Registry : 74844
FAISS Index Vectors      : 74844
BM25 Corpus Size         : 74844

All indexes loaded and verified successfully!


In [2]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print(f"\nLoading Bi-Encoder ({BI_ENCODER_NAME}) on {DEVICE}...")
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)

print(f"\nLoading Cross-Encoder ({CROSS_ENCODER_NAME}) on {DEVICE}...")
cross_encoder = CrossEncoder(
    CROSS_ENCODER_NAME,
    max_length=512,
    device=DEVICE
)

print("\nBoth models loaded successfully!")

GPU: NVIDIA GeForce RTX 5050 Laptop GPU

Loading Bi-Encoder (sentence-transformers/all-MiniLM-L6-v2) on cuda...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading Cross-Encoder (cross-encoder/ms-marco-MiniLM-L-6-v2) on cuda...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Both models loaded successfully!


In [3]:
def tokenize(text):
    return re.findall(r"\b[\w-]+\b", text.lower())


def hybrid_retrieve_and_rerank(query, candidate_k=20, final_top_k=5, rrf_k=60):

    """
    Stage 1: Dense (FAISS) + Sparse (BM25) Retrieval
    Stage 2: Reciprocal Rank Fusion (RRF)
    Stage 3: Cross-Encoder Re-ranking
    """

    if not query.strip():
        raise ValueError("Query cannot be empty.")

    candidate_k = min(candidate_k, len(chunks))

    # 1. Dense Retrieval
    query_embedding = bi_encoder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    dense_scores, dense_indices = faiss_index.search(
        query_embedding, candidate_k
    )

    dense_ranked_ids = dense_indices[0].tolist()

    # 2. Sparse Retrieval
    tokenized_query = tokenize(query)

    bm25_scores = bm25_index.get_scores(tokenized_query)

    sparse_ranked_ids = np.argsort(bm25_scores)[::-1][:candidate_k].tolist()

    # 3. Reciprocal Rank Fusion
    rrf_scores = {}

    for rank, doc_idx in enumerate(dense_ranked_ids):
        if doc_idx >= 0:
            rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + (
                1.0 / (rrf_k + rank + 1)
            )

    for rank, doc_idx in enumerate(sparse_ranked_ids):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + (
            1.0 / (rrf_k + rank + 1)
        )

    fused_ids = sorted(
        rrf_scores,
        key=lambda idx: rrf_scores[idx],
        reverse=True
    )[:candidate_k]

    if not fused_ids:
        return []

    # 4. Cross-Encoder Re-ranking
    candidate_pairs = [
        (query, chunks[idx]["text"])
        for idx in fused_ids
    ]

    cross_scores = cross_encoder.predict(
        candidate_pairs,
        batch_size=16
    )

    # 5. Prepare Final Results
    reranked_results = []

    for idx, ce_score in zip(fused_ids, cross_scores):
        chunk_data = chunks[idx].copy()

        chunk_data["rrf_score"] = round(
            float(rrf_scores[idx]), 6
        )

        chunk_data["cross_encoder_score"] = round(
            float(ce_score), 4
        )

        reranked_results.append(chunk_data)

    reranked_results.sort(
        key=lambda x: x["cross_encoder_score"],
        reverse=True
    )

    return reranked_results[:final_top_k]

In [4]:
import time
import re
test_queries = [
    "What is the AISHE code of VNIT Nagpur?",
    "Engineering colleges in Maharashtra",
    "Government colleges in Telangana",
    "Universities located in Uttar Pradesh",
    "Which university is affiliated with colleges in Nagpur?",
    "Private unaided colleges in Maharashtra",
    "Research institutes under the Ministry of Agriculture",
    "Institutions established after 2010 in Rajasthan"
]

for q in test_queries:
    start_time = time.time()

    results = hybrid_retrieve_and_rerank(
        q,
        candidate_k=20,
        final_top_k=5
    )

    latency_ms = (time.time() - start_time) * 1000

    print("=" * 100)
    print(f"QUERY: {q}")
    print(f"Latency: {latency_ms:.2f} ms")
    print("=" * 100)

    if not results:
        print("No results found.\n")
        continue

    for rank, res in enumerate(results, start=1):
        metadata = res["metadata"]

        print(f"\nRank: {rank}")
        print(f"Chunk ID: {res['chunk_id']}")
        print(f"Institution: {metadata['institution_name']}")
        print(f"AISHE Code: {metadata['aishe_code']}")
        print(f"Category: {metadata['institution_category']}")
        print(f"State: {metadata['state']}")
        print(f"District: {metadata['district']}")
        print(f"RRF Score: {res['rrf_score']}")
        print(f"Cross-Encoder Score: {res['cross_encoder_score']}")
        print("-" * 80)

QUERY: What is the AISHE code of VNIT Nagpur?
Latency: 3133.73 ms

Rank: 1
Chunk ID: INST-074256
Institution: Visvesvaraya National Institute of Technology Nagpur
AISHE Code: U-0334
Category: Vidya Lakshmi
State: Maharashtra
District: Nagpur
RRF Score: 0.032002
Cross-Encoder Score: 6.9674
--------------------------------------------------------------------------------

Rank: 2
Chunk ID: INST-055993
Institution: Visvesvaraya National Institute of Technology, Nagpur
AISHE Code: U-0334
Category: Universities
State: Maharashtra
District: Nagpur
RRF Score: 0.032787
Cross-Encoder Score: 6.9296
--------------------------------------------------------------------------------

Rank: 3
Chunk ID: INST-074145
Institution: INDIAN INSTITUTE OF INFORMATION TECHNOLOGY, NAGPUR
AISHE Code: U-0841
Category: Vidya Lakshmi
State: Maharashtra
District: Nagpur
RRF Score: 0.014706
Cross-Encoder Score: 5.5431
--------------------------------------------------------------------------------

Rank: 4
Chunk ID: IN

In [5]:
from typing import List

from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_ollama import ChatOllama


class CampusNLPHybridRetriever(BaseRetriever):

    candidate_k: int = 20
    final_top_k: int = 5

    def _get_relevant_documents(self, query: str) -> List[Document]:

        raw_results = hybrid_retrieve_and_rerank(
            query=query,
            candidate_k=self.candidate_k,
            final_top_k=self.final_top_k
        )

        docs = []

        for idx, result in enumerate(raw_results, start=1):

            metadata = result["metadata"]

            docs.append(
                Document(
                    page_content=result["text"],
                    metadata={
                        "source_num": idx,
                        "chunk_id": result["chunk_id"],
                        "aishe_code": metadata["aishe_code"],
                        "institution_name": metadata["institution_name"],
                        "institution_category": metadata["institution_category"],
                        "state": metadata["state"],
                        "district": metadata["district"],
                        "website": metadata["website"],
                        "rrf_score": result["rrf_score"],
                        "cross_encoder_score": result["cross_encoder_score"]
                    }
                )
            )

        return docs


# Initialize Hybrid Retriever

retriever = CampusNLPHybridRetriever(
    candidate_k=20,
    final_top_k=5
)


# Initialize Local LLM

llm = ChatOllama(
    model="qwen2.5:latest",
    temperature=0.1,
    num_ctx=4096
)


# Grounded Prompt

prompt = ChatPromptTemplate.from_messages([

    ("system", """
You are CampusNLP, an intelligent institutional information
assistant powered by AISHE (All India Survey on Higher Education)
institutional directory data.

Your responsibility is to answer questions about Indian higher
education institutions using only the retrieved AISHE records.

STRICT GROUNDING RULES:

1. Use only information explicitly available in the retrieved context.
   Never invent institutional details or use unsupported external knowledge.

2. If the retrieved records do not contain sufficient information,
   clearly state that the available AISHE records do not provide
   the requested information.

3. Do not assume missing information. Treat unavailable fields as
   unknown.

4. Preserve AISHE codes, institution names, establishment years,
   management types, and other values exactly as provided.

5. Distinguish between colleges, universities, standalone institutions,
   research institutes, and Vidya Lakshmi records.

6. The same AISHE code may appear in multiple source datasets.
   Do not treat repeated records as separate institutions without
   acknowledging the overlap.

7. Do not claim that retrieved results represent every institution
   satisfying a query. They are ranked retrieval results unless
   exhaustive filtering has been explicitly performed.

ANSWERING GUIDELINES:

1. Understand the complete intent of the user's question.

2. Answer directly and clearly.

3. For institution-specific questions, include relevant details such as:
   - Institution name
   - AISHE code
   - Institution category
   - State and district
   - Establishment year
   - Management type
   - Affiliated university
   - Website

   Include only fields available in the retrieved context.

4. For questions involving multiple institutions, use a Markdown table
   when appropriate.

5. For questions involving counts, rankings, or statistical summaries,
   do not calculate totals from a limited set of retrieved results.
   State when exhaustive structured-data processing is required.

6. Do not answer questions about faculty, directors, contact numbers,
   hostel fees, course syllabi, or other information that is not
   contained in the AISHE records.

SOURCE ATTRIBUTION:

1. Support factual claims using inline citations such as [Source 1].

2. Cite the relevant source number for each institution.

3. At the end, include a Sources section containing:
   - Institution name
   - AISHE code
   - Institution category

4. Include a website only when it is present in the retrieved metadata.

5. Never invent source numbers, website URLs, or institutional details.

RESPONSE STYLE:

- Use clear, professional English.
- Prefer concise and factual answers.
- Use tables for institution comparisons.
- Clearly communicate missing information.
- Prioritize accuracy over completeness.

Your objective is to provide reliable, grounded answers from
AISHE institutional directory records.
"""),

    ("human", """
RETRIEVED AISHE CONTEXT:

{context}

USER QUESTION:

{question}

Provide an accurate answer using the retrieved records and
appropriate source citations.
""")
])


# Format Retrieved Documents

def format_docs_with_sources(docs: List[Document]) -> str:

    blocks = []

    for doc in docs:

        source_num = doc.metadata["source_num"]

        blocks.append(
            f"[Source {source_num}]\n"
            f"{doc.page_content}"
        )

    return "\n\n---\n\n".join(blocks)


# Build RAG Chain

rag_chain = (
    RunnableParallel(
        docs=retriever,
        question=RunnablePassthrough()
    )
    .assign(
        answer=(
            RunnablePassthrough.assign(
                context=lambda x: format_docs_with_sources(x["docs"])
            )
            | prompt
            | llm
            | StrOutputParser()
        )
    )
)

print("CampusNLP Hybrid RAG Chain initialized successfully!")

CampusNLP Hybrid RAG Chain initialized successfully!


In [6]:
test_queries = [
    "What is the AISHE code of VNIT Nagpur?",
    "List engineering colleges in Maharashtra.",
    "Which universities are located in Uttar Pradesh?",
    "Find government colleges in Telangana.",
    "Which university is affiliated with colleges in Nagpur?",
    "Find private unaided colleges in Maharashtra.",
    "What research institutes are listed under the Ministry of Agriculture?",
    "Find institutions established after 2010 in Rajasthan."
]

for query in test_queries:
    print("=" * 100)
    print("QUESTION:", query)

    response = rag_chain.invoke(query)

    print("\nANSWER:")
    print(response["answer"])

    print("\nRETRIEVED SOURCES:")
    for doc in response["docs"]:
        print(
            f"- {doc.metadata['institution_name']} "
            f"| AISHE: {doc.metadata['aishe_code']} "
            f"| Category: {doc.metadata['institution_category']}"
        )

    print()

QUESTION: What is the AISHE code of VNIT Nagpur?

ANSWER:
The AISHE code of Visvesvaraya National Institute of Technology, Nagpur (VNIT Nagpur) is U-0334. This information is sourced from [Source 1] and [Source 2].

Sources:
- Visvesvaraya National Institute of Technology Nagpur | AISHE Code: U-0334
- Visvesvaraya National Institute of Technology, Nagpur | AISHE Code: U-0334

RETRIEVED SOURCES:
- Visvesvaraya National Institute of Technology Nagpur | AISHE: U-0334 | Category: Vidya Lakshmi
- Visvesvaraya National Institute of Technology, Nagpur | AISHE: U-0334 | Category: Universities
- INDIAN INSTITUTE OF INFORMATION TECHNOLOGY, NAGPUR | AISHE: U-0841 | Category: Vidya Lakshmi
- JAWAHARLAL NEHRU ALUMINIUM RESEARCH DEVELOPMENT AND DESIGN CENTRE NAGPUR | AISHE: R-10242 | Category: Research
- SYMBIOSIS INSTITUTE OF TECHNOLOGY, NAGPUR | AISHE: C-66696 | Category: Vidya Lakshmi

QUESTION: List engineering colleges in Maharashtra.

ANSWER:
Here is a list of engineering colleges in Maharasht

# Evaluation

In [7]:
import pandas as pd

evaluation_df = pd.DataFrame([
    {
        "chunk_id": chunk["chunk_id"],
        "aishe_code": chunk["metadata"]["aishe_code"],
        "institution_name": chunk["metadata"]["institution_name"],
        "category": chunk["metadata"]["institution_category"],
        "state": chunk["metadata"]["state"],
        "district": chunk["metadata"]["district"],
        "establishment_year": chunk["metadata"]["establishment_year"],
        "management_type": chunk["metadata"]["management_type"]
    }
    for chunk in chunks
])

print("Evaluation records:", len(evaluation_df))
display(evaluation_df.head())

Evaluation records: 74844


,chunk_id,aishe_code,institution_name,category,state,district,establishment_year,management_type
0,INST-000001,C-59634,"061-PT. SRI JAGANNATH PRASAD MAHAVIDHYALAYA,DA...",Colleges,Uttar Pradesh,Agra,2016,Private Un-Aided
1,INST-000002,C-8913,100002-J.D.M.V.P. SAMAJ'S SHRI S. S. PATIL ART...,Colleges,Maharashtra,Jalgaon,1972,Private Aided (Government Aided)
2,INST-000003,C-8922,100004-Khandesh College Education Society'S Co...,Colleges,Maharashtra,Jalgaon,1965,Private Aided (Government Aided)
3,INST-000004,C-9030,"100005-KCES'S S.S.MANIYAR LAW COLLEGE, JALGAON.",Colleges,Maharashtra,Jalgaon,1970,Private Aided (Government Aided)
4,INST-000005,C-9025,100006-KRIDA RASIK EDUCATION SOCIETY INSTITUTE...,Colleges,Maharashtra,Jalgaon,1983,Private Aided (Government Aided)


In [8]:
evaluation_queries = [
    {
        "query": "What is the AISHE code of VNIT Nagpur?",
        "relevant_chunk_ids": []
    },
    {
        "query": "List engineering colleges in Maharashtra.",
        "relevant_chunk_ids": []
    },
    {
        "query": "Which universities are located in Uttar Pradesh?",
        "relevant_chunk_ids": []
    },
    {
        "query": "Find government colleges in Telangana.",
        "relevant_chunk_ids": []
    },
    {
        "query": "Which university is affiliated with colleges in Nagpur?",
        "relevant_chunk_ids": []
    },
    {
        "query": "Find private unaided colleges in Maharashtra.",
        "relevant_chunk_ids": []
    },
    {
        "query": "What research institutes are listed under the Ministry of Agriculture?",
        "relevant_chunk_ids": []
    },
    {
        "query": "Find institutions established after 2010 in Rajasthan.",
        "relevant_chunk_ids": []
    },
    {
        "query": "List standalone institutions in Karnataka.",
        "relevant_chunk_ids": []
    },
    {
        "query": "Find universities established after 2000 in Maharashtra.",
        "relevant_chunk_ids": []
    }
]

print("Total evaluation queries:", len(evaluation_queries))

for i, item in enumerate(evaluation_queries, start=1):
    print(f"{i}. {item['query']}")

Total evaluation queries: 10
1. What is the AISHE code of VNIT Nagpur?
2. List engineering colleges in Maharashtra.
3. Which universities are located in Uttar Pradesh?
4. Find government colleges in Telangana.
5. Which university is affiliated with colleges in Nagpur?
6. Find private unaided colleges in Maharashtra.
7. What research institutes are listed under the Ministry of Agriculture?
8. Find institutions established after 2010 in Rajasthan.
9. List standalone institutions in Karnataka.
10. Find universities established after 2000 in Maharashtra.


In [9]:
def find_candidates(keyword, state=None, category=None):
    results = evaluation_df[
        evaluation_df["institution_name"].str.contains(
            keyword, case=False, na=False
        )
    ]

    if state:
        results = results[
            results["state"].str.contains(state, case=False, na=False)
        ]

    if category:
        results = results[
            results["category"].str.contains(category, case=False, na=False)
        ]

    return results[
        [
            "chunk_id",
            "aishe_code",
            "institution_name",
            "category",
            "state",
            "district"
        ]
    ].reset_index(drop=True)


display(find_candidates("Visvesvaraya National Institute of Technology"))

,chunk_id,aishe_code,institution_name,category,state,district
0,INST-055993,U-0334,"Visvesvaraya National Institute of Technology,...",Universities,Maharashtra,Nagpur
1,INST-074256,U-0334,Visvesvaraya National Institute of Technology ...,Vidya Lakshmi,Maharashtra,Nagpur


In [10]:
vnit_candidates = evaluation_df[
    evaluation_df["institution_name"].str.contains(
        "Visvesvaraya National Institute of Technology",
        case=False,
        na=False
    )
]

for _, row in vnit_candidates.iterrows():
    print("-" * 80)
    print("Chunk ID:", row["chunk_id"])
    print("AISHE Code:", row["aishe_code"])
    print("Institution:", row["institution_name"])
    print("Category:", row["category"])
    print("State:", row["state"])
    print("District:", row["district"])

--------------------------------------------------------------------------------
Chunk ID: INST-055993
AISHE Code: U-0334
Institution: Visvesvaraya National Institute of Technology, Nagpur
Category: Universities
State: Maharashtra
District: Nagpur
--------------------------------------------------------------------------------
Chunk ID: INST-074256
AISHE Code: U-0334
Institution: Visvesvaraya National Institute of Technology Nagpur
Category: Vidya Lakshmi
State: Maharashtra
District: Nagpur


In [11]:
evaluation_queries[0]["relevant_chunk_ids"] = (
    vnit_candidates["chunk_id"].tolist()
)

print("Question:", evaluation_queries[0]["query"])
print("Relevant chunk IDs:", evaluation_queries[0]["relevant_chunk_ids"])
print("Number of relevant records:", len(evaluation_queries[0]["relevant_chunk_ids"]))

Question: What is the AISHE code of VNIT Nagpur?
Relevant chunk IDs: ['INST-055993', 'INST-074256']
Number of relevant records: 2


In [12]:
engineering_candidates = evaluation_df[
    (
        evaluation_df["institution_name"].str.contains(
            "engineering", case=False, na=False
        )
    )
    &
    (
        evaluation_df["state"].str.contains(
            "Maharashtra", case=False, na=False
        )
    )
    &
    (
        evaluation_df["category"].str.contains(
            "Colleges", case=False, na=False
        )
    )
]

print("Candidate records found:", len(engineering_candidates))

display(
    engineering_candidates[
        [
            "chunk_id",
            "aishe_code",
            "institution_name",
            "state",
            "district",
            "category"
        ]
    ].head(20)
)

Candidate records found: 305


,chunk_id,aishe_code,institution_name,state,district,category
5,INST-000006,C-8977,100007-SSBT'S COLLEGE OF ENGINEERING & TECHNOL...,Maharashtra,Jalgaon,Colleges
12,INST-000013,C-8836,"100063-GF'S GODAVARI COLLEGE OF ENGINEERING, J...",Maharashtra,Jalgaon,Colleges
26,INST-000027,C-8830,110059-HINDI SEVA MANDAL'S SHRI.SANT GADGE BAB...,Maharashtra,Jalgaon,Colleges
43,INST-000044,C-9012,150027-TMES'S J.T.MAHAJAN COLLEGE OF ENGINEERI...,Maharashtra,Jalgaon,Colleges
66,INST-000067,C-8874,200004-SSVPS'S BAPUSAHEB SHIVAJIRAO DEORE COLL...,Maharashtra,Dhule,Colleges
75,INST-000076,C-8846,"200025-NES'S GANGAMAI COLLEGE OF ENGINEERING, ...",Maharashtra,Dhule,Colleges
98,INST-000099,C-8912,230032-PSGVPM'S D.N.PATEL COLLEGE OF ENGINEERI...,Maharashtra,Nandurbar,Colleges
130,INST-000131,C-72853,3143-THAKUR SHYAMNARAYAN ENGINEERING COLLEGE,Maharashtra,Mumbai Suburban,Colleges
338,INST-000339,C-11223,Abhinav Education Society College of Engineeri...,Maharashtra,Satara,Colleges
707,INST-000708,C-34488,Adarsh Shikshan Prasarak Mandal's K.T. Patil C...,Maharashtra,Dharashiv,Colleges


In [13]:
print("Total engineering candidates:", len(engineering_candidates))

display(
    engineering_candidates[
        [
            "chunk_id",
            "aishe_code",
            "institution_name",
            "district"
        ]
    ]
)

Total engineering candidates: 305


,chunk_id,aishe_code,institution_name,district
5,INST-000006,C-8977,100007-SSBT'S COLLEGE OF ENGINEERING & TECHNOL...,Jalgaon
12,INST-000013,C-8836,"100063-GF'S GODAVARI COLLEGE OF ENGINEERING, J...",Jalgaon
26,INST-000027,C-8830,110059-HINDI SEVA MANDAL'S SHRI.SANT GADGE BAB...,Jalgaon
43,INST-000044,C-9012,150027-TMES'S J.T.MAHAJAN COLLEGE OF ENGINEERI...,Jalgaon
66,INST-000067,C-8874,200004-SSVPS'S BAPUSAHEB SHIVAJIRAO DEORE COLL...,Dhule
...,...,...,...,...
54259,INST-054260,C-33830,"Xavier Institute of Engineering, Mahim Causewa...",Mumbai
54275,INST-054276,C-33958,Yadavrao Tasgaonkar College of Engineering and...,Raigad
54277,INST-054278,C-33938,Yadavrao Tasgoankar Institute of Engineering &...,Raigad
54376,INST-054377,C-18254,"Yashwantrao Chavan Engineering College, Wanado...",Nagpur


In [14]:
engineering_sample = engineering_candidates.sample(
    n=min(10, len(engineering_candidates)),
    random_state=42
)

display(
    engineering_sample[
        [
            "chunk_id",
            "aishe_code",
            "institution_name",
            "district",
            "management_type"
        ]
    ].reset_index(drop=True)
)

,chunk_id,aishe_code,institution_name,district,management_type
0,INST-031655,C-41294,Padamshri. Dr.Vitthalrao Vikhe Patil Foundatio...,Ahilyanagar,Private Un-Aided
1,INST-027032,C-75022,"MANGALDEEP COLLEGE OF ENGINEERING, NIPANI",Chhatrapati Sambhajinagar,Private Un-Aided
2,INST-020413,C-44591,"Jai Hind College of Engineering, Kuran, Tal. J...",Pune,Private Un-Aided
3,INST-043653,C-7470,Shri Shivaji Institute of Engineering and Mana...,Parbhani,Private Un-Aided
4,INST-010675,C-48655,DNYANSHREE INSTITUTE OF ENGINEERING AND TECHNO...,Satara,Private Un-Aided
5,INST-000708,C-34488,Adarsh Shikshan Prasarak Mandal's K.T. Patil C...,Dharashiv,Local Body
6,INST-021206,C-74697,JIJAU INSTITUTE OF ENGINEERING TECHNOLOGY AND ...,Nanded,Private Un-Aided
7,INST-027264,C-41854,"Maratha Vidya Prasarak Samajs Law College , U...",Nashik,University
8,INST-028780,C-41607,Modern Education Society's College of Engineer...,Pune,Private Un-Aided
9,INST-005345,C-69992,Bharati Vidyapeeth Deemed to be University Dep...,Raigad,Private Un-Aided


In [15]:
for i, (_, row) in enumerate(engineering_sample.iterrows(), start=1):
    print(f"\n{i}. {row['institution_name']}")
    print("   Chunk ID:", row["chunk_id"])
    print("   AISHE Code:", row["aishe_code"])
    print("   District:", row["district"])
    print("   Management:", row["management_type"])


1. Padamshri. Dr.Vitthalrao Vikhe Patil Foundations Padamshri. Dr.Vitthalrao Vikhe Patil College of Engineering, Vilad Ghat, Ahmednagar 414111
   Chunk ID: INST-031655
   AISHE Code: C-41294
   District: Ahilyanagar
   Management: Private Un-Aided

2. MANGALDEEP COLLEGE OF ENGINEERING, NIPANI
   Chunk ID: INST-027032
   AISHE Code: C-75022
   District: Chhatrapati Sambhajinagar
   Management: Private Un-Aided

3. Jai Hind College of Engineering, Kuran, Tal. Junnar, Dist. Pune
   Chunk ID: INST-020413
   AISHE Code: C-44591
   District: Pune
   Management: Private Un-Aided

4. Shri Shivaji Institute of Engineering and Management Studies, Parbhani
   Chunk ID: INST-043653
   AISHE Code: C-7470
   District: Parbhani
   Management: Private Un-Aided

5. DNYANSHREE INSTITUTE OF ENGINEERING AND TECHNOLOGY, SATARA
   Chunk ID: INST-010675
   AISHE Code: C-48655
   District: Satara
   Management: Private Un-Aided

6. Adarsh Shikshan Prasarak Mandal's K.T. Patil College of Engineering and Techn

In [16]:
remaining_engineering = engineering_candidates[
    ~engineering_candidates["chunk_id"].isin(
        engineering_sample["chunk_id"]
    )
]

engineering_sample_2 = remaining_engineering.sample(
    n=min(10, len(remaining_engineering)),
    random_state=100
)

for i, (_, row) in enumerate(engineering_sample_2.iterrows(), start=11):
    print(f"\n{i}. {row['institution_name']}")
    print("   Chunk ID:", row["chunk_id"])
    print("   AISHE Code:", row["aishe_code"])
    print("   District:", row["district"])
    print("   Management:", row["management_type"])


11. SKN SINHGAD COLLEGE OF ENGINEERING KORTI TAL PANDHARPUR DIST-SOLAPUR
   Chunk ID: INST-044787
   AISHE Code: C-15745
   District: Solapur
   Management: Private Un-Aided

12. SJVPM COLLEGE OF ENGINEERING, CHINCHWAD,PUNE
   Chunk ID: INST-044723
   AISHE Code: C-77898
   District: Pune
   Management: Private Un-Aided

13. Sharad Institute of Technology College of Engineering, Gat
   Chunk ID: INST-040267
   AISHE Code: C-11110
   District: Kolhapur
   Management: Private Un-Aided

14. PDEAs A G Awate College of Engineering, Hadapsar, Pune 28
   Chunk ID: INST-032246
   AISHE Code: C-41924
   District: Pune
   Management: Private Un-Aided

15. Pimpri Chinchwad Education Trust's Pimpri Chinchwad College of Engineering, Nigdi, Pune
   Chunk ID: INST-032451
   AISHE Code: C-41991
   District: Pune
   Management: Private Un-Aided

16. Vidya Vikas Education Trust, Universal College of Engineering, Kaman Bhiwandi Road, Survey No. 146 (Part), Village Kaman, Vasai (E) -401212, Tal. Vasai. D

In [17]:
verified_engineering_ids = [
    "INST-031655",
    "INST-027032",
    "INST-020413",
    "INST-043653",
    "INST-010675",
    "INST-000708",
    "INST-021206",
    "INST-028780",
    "INST-005345",
    "INST-044787",
    "INST-044723",
    "INST-040267",
    "INST-032246",
    "INST-032451",
    "INST-052999",
    "INST-026978",
    "INST-014507",
    "INST-003399",
    "INST-041547"
]

evaluation_queries[1]["relevant_chunk_ids"] = verified_engineering_ids

print("Question:", evaluation_queries[1]["query"])
print("Relevant records:", len(verified_engineering_ids))
print("Unique chunk IDs:", len(set(verified_engineering_ids)))

assert len(verified_engineering_ids) == len(set(verified_engineering_ids))
assert set(verified_engineering_ids).issubset(
    set(evaluation_df["chunk_id"])
)

print("Ground-truth labels validated.")

Question: List engineering colleges in Maharashtra.
Relevant records: 19
Unique chunk IDs: 19
Ground-truth labels validated.


In [18]:
up_university_candidates = evaluation_df[
    (
        evaluation_df["category"].str.contains(
            "Universities", case=False, na=False
        )
    )
    &
    (
        evaluation_df["state"].str.contains(
            "Uttar Pradesh", case=False, na=False
        )
    )
]

print("Candidate universities:", len(up_university_candidates))

display(
    up_university_candidates[
        [
            "chunk_id",
            "aishe_code",
            "institution_name",
            "state",
            "district"
        ]
    ].reset_index(drop=True)
)

Candidate universities: 126


,chunk_id,aishe_code,institution_name,state,district
0,INST-054576,U-0713,ACADEMY OF SCIENTIFIC AND INNOVATIVE RESEARCH,Uttar Pradesh,Ghaziabad
1,INST-054578,U-0531,Acharya Narendra Deva University of Agricultur...,Uttar Pradesh,Ayodhya
2,INST-054586,U-1271,AGRAWAN HERITAGE UNIVERSITY,Uttar Pradesh,Agra
3,INST-054599,U-0496,"Aligarh Muslim University, Aligarh",Uttar Pradesh,Aligarh
4,INST-054606,U-1180,All India Institute of Medical Science Gorakhpur,Uttar Pradesh,Gorakhpur
...,...,...,...,...,...
121,INST-055959,U-0882,"UTTAR PRADESH UNIVERSITY OF MEDICAL SCIENCES,",Uttar Pradesh,Etawah
122,INST-055963,U-1348,Varun Arjun University,Uttar Pradesh,Shahjahanpur
123,INST-055967,U-0550,"Veer Bahadur Singh Purvanchal University, Jaunpur",Uttar Pradesh,Jaunpur
124,INST-055980,U-1385,Vidya University,Uttar Pradesh,Meerut


In [19]:
up_university_candidates = evaluation_df[
    evaluation_df["category"].str.contains(
        "Universities", case=False, na=False
    )
    &
    evaluation_df["state"].str.contains(
        "Uttar Pradesh", case=False, na=False
    )
]

print("Candidate universities:", len(up_university_candidates))

display(
    up_university_candidates[
        ["chunk_id", "aishe_code", "institution_name", "state"]
    ].sample(
        n=min(5, len(up_university_candidates)),
        random_state=42
    ).reset_index(drop=True)
)

Candidate universities: 126


,chunk_id,aishe_code,institution_name,state
0,INST-055342,U-0952,MAHARISHI UNIVERSITY OF INFORMATION TECHNOLOGY,Uttar Pradesh
1,INST-054794,U-0504,Chandra Shekhar Azad University of Agriculture...,Uttar Pradesh
2,INST-055915,U-1214,UNITED UNIVERSITY,Uttar Pradesh
3,INST-055297,U-1407,Maa Pateswari University,Uttar Pradesh
4,INST-055635,U-1198,Rani Lakshmi Bai Central Agricultural University,Uttar Pradesh


In [20]:
telangana_govt_candidates = evaluation_df[
    evaluation_df["category"].str.contains(
        "Colleges", case=False, na=False
    )
    &
    evaluation_df["state"].str.contains(
        "Telangana", case=False, na=False
    )
    &
    evaluation_df["management_type"].str.contains(
        "Government", case=False, na=False
    )
]

print("Candidate records:", len(telangana_govt_candidates))

display(
    telangana_govt_candidates[
        [
            "chunk_id",
            "aishe_code",
            "institution_name",
            "state",
            "management_type"
        ]
    ].sample(
        n=min(5, len(telangana_govt_candidates)),
        random_state=42
    ).reset_index(drop=True)
)

Candidate records: 422


,chunk_id,aishe_code,institution_name,state,management_type
0,INST-015551,C-70981,"GOVERNMENT MEDICAL COLLEGE, KHAMMAM",Telangana,State Government
1,INST-047803,C-30467,"Sri Venkata Sai College of Nursing, Mahabubnagar",Telangana,Private Aided (Government Aided)
2,INST-016503,C-53331,"Govt. Degree College, Bheemgal, Nizamabad (5073)",Telangana,State Government
3,INST-050657,C-57965,TELANGANA SOCIAL WELFARE RESIDENTIAL DEGREE CO...,Telangana,State Government
4,INST-053728,C-26043,Vivekananda Govt. Degree College,Telangana,State Government


In [21]:
up_sample = up_university_candidates.sample(
    n=min(5, len(up_university_candidates)),
    random_state=42
)

telangana_sample = telangana_govt_candidates.sample(
    n=min(5, len(telangana_govt_candidates)),
    random_state=42
)

print("Uttar Pradesh universities:")
display(up_sample[["chunk_id", "aishe_code", "institution_name"]])

print("\nTelangana government colleges:")
display(telangana_sample[["chunk_id", "aishe_code", "institution_name"]])

Uttar Pradesh universities:


,chunk_id,aishe_code,institution_name
55341,INST-055342,U-0952,MAHARISHI UNIVERSITY OF INFORMATION TECHNOLOGY
54793,INST-054794,U-0504,Chandra Shekhar Azad University of Agriculture...
55914,INST-055915,U-1214,UNITED UNIVERSITY
55296,INST-055297,U-1407,Maa Pateswari University
55634,INST-055635,U-1198,Rani Lakshmi Bai Central Agricultural University



Telangana government colleges:


,chunk_id,aishe_code,institution_name
15550,INST-015551,C-70981,"GOVERNMENT MEDICAL COLLEGE, KHAMMAM"
47802,INST-047803,C-30467,"Sri Venkata Sai College of Nursing, Mahabubnagar"
16502,INST-016503,C-53331,"Govt. Degree College, Bheemgal, Nizamabad (5073)"
50656,INST-050657,C-57965,TELANGANA SOCIAL WELFARE RESIDENTIAL DEGREE CO...
53727,INST-053728,C-26043,Vivekananda Govt. Degree College


In [22]:
# Reproducible, data-derived benchmark
#
# Every query below has exhaustive relevance labels derived from the
# AISHE fields.  We deliberately retain only filters with <= 20 matches,
# because this notebook evaluates a top-20 retriever.
BENCHMARK_SEED = 42
MAX_RELEVANT_DOCS = 20
QUERIES_PER_FAMILY = 24

benchmark_df = pd.DataFrame([
    {"chunk_id": chunk["chunk_id"], **chunk["metadata"]}
    for chunk in chunks
])
benchmark_df["establishment_year_num"] = pd.to_numeric(
    benchmark_df["establishment_year"], errors="coerce"
)

def add_benchmark_query(items, query, family, matching_rows):
    relevant_ids = matching_rows["chunk_id"].drop_duplicates().tolist()
    if 1 <= len(relevant_ids) <= MAX_RELEVANT_DOCS:
        items.append({
            "query": query,
            "query_family": family,
            "relevant_chunk_ids": relevant_ids
        })

evaluation_queries = []

# 1. Exact entity lookup: balanced across source categories.
entity_pool = benchmark_df.dropna(subset=["institution_name", "aishe_code"]).copy()
entity_pool = entity_pool.groupby("aishe_code").filter(
    lambda group: len(group) <= MAX_RELEVANT_DOCS
)
entity_sample = (
    entity_pool.groupby("institution_category", group_keys=False)
    .apply(lambda group: group.sample(min(5, len(group)), random_state=BENCHMARK_SEED))
    .sample(frac=1, random_state=BENCHMARK_SEED)
    .head(QUERIES_PER_FAMILY)
)
for _, row in entity_sample.iterrows():
    matches = benchmark_df[benchmark_df["aishe_code"].eq(row["aishe_code"])]
    add_benchmark_query(
        evaluation_queries,
        f"What is the AISHE code of {row['institution_name']}?",
        "entity_lookup",
        matches
    )

# 2. Category + state filters.
category_state_groups = []
for (category, state), group in benchmark_df.dropna(subset=["institution_category", "state"]).groupby(
    ["institution_category", "state"]
):
    if 1 <= len(group) <= MAX_RELEVANT_DOCS:
        category_state_groups.append((category, state, group))
for category, state, group in pd.Series(category_state_groups).sample(
    min(QUERIES_PER_FAMILY, len(category_state_groups)), random_state=BENCHMARK_SEED
):
    add_benchmark_query(
        evaluation_queries, f"List {category.lower()} in {state}.",
        "category_state", group
    )

# 3. Category + state + management filters.
management_groups = []
for (category, state, management), group in benchmark_df.dropna(
    subset=["institution_category", "state", "management_type"]
).groupby(["institution_category", "state", "management_type"]):
    if 1 <= len(group) <= MAX_RELEVANT_DOCS:
        management_groups.append((category, state, management, group))
for category, state, management, group in pd.Series(management_groups).sample(
    min(QUERIES_PER_FAMILY, len(management_groups)), random_state=BENCHMARK_SEED + 1
):
    add_benchmark_query(
        evaluation_queries, f"Find {management} {category.lower()} in {state}.",
        "category_state_management", group
    )

# 4. Establishment-year queries test numeric facts expressed in text.
year_groups = []
year_df = benchmark_df.dropna(subset=["institution_category", "state", "establishment_year_num"])
for (category, state, year), group in year_df.groupby(
    ["institution_category", "state", "establishment_year_num"]
):
    if 1 <= len(group) <= MAX_RELEVANT_DOCS:
        year_groups.append((category, state, int(year), group))
for category, state, year, group in pd.Series(year_groups).sample(
    min(QUERIES_PER_FAMILY, len(year_groups)), random_state=BENCHMARK_SEED + 2
):
    add_benchmark_query(
        evaluation_queries,
        f"Which {category.lower()} in {state} were established in {year}?",
        "category_state_year", group
    )

assert evaluation_queries
all_chunk_ids = set(benchmark_df["chunk_id"])
for item in evaluation_queries:
    assert len(item["relevant_chunk_ids"]) == len(set(item["relevant_chunk_ids"]))
    assert set(item["relevant_chunk_ids"]).issubset(all_chunk_ids)

benchmark_output = DATA_PROCESSED_DIR / "hybrid_evaluation_queries.json"
with open(benchmark_output, "w", encoding="utf-8") as f:
    json.dump(evaluation_queries, f, ensure_ascii=False, indent=2)

benchmark_summary = pd.DataFrame(evaluation_queries)
diagnostic_queries = (
    benchmark_summary.groupby("query_family", group_keys=False)
    .apply(lambda group: group.sample(min(2, len(group)), random_state=BENCHMARK_SEED))
    .to_dict("records")
)
llm_eval_queries = (
    benchmark_summary.groupby("query_family", group_keys=False)
    .apply(lambda group: group.sample(min(3, len(group)), random_state=BENCHMARK_SEED + 10))
    .to_dict("records")
)
performance_eval_queries = (
    benchmark_summary.groupby("query_family", group_keys=False)
    .apply(lambda group: group.sample(min(5, len(group)), random_state=BENCHMARK_SEED + 20))
    .to_dict("records")
)
print(f"Generated {len(evaluation_queries)} evaluation queries.")
display(benchmark_summary.groupby("query_family").agg(
    queries=("query", "count"),
    avg_relevant=("relevant_chunk_ids", lambda ids: round(np.mean([len(x) for x in ids]), 2))
))
print(f"Saved benchmark: {benchmark_output}")

Generated 96 evaluation queries.


,queries,avg_relevant
query_family,,
category_state,24,6.12
category_state_management,24,8.38
category_state_year,24,3.04
entity_lookup,24,1.21


Saved benchmark: E:\Rasengan\College-Knowledge-Search\data\processed\hybrid_evaluation_queries.json


In [23]:
import numpy as np
import pandas as pd

def calculate_retrieval_metrics(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)
    retrieved_k = retrieved_ids[:k]

    relevance = [
        1 if chunk_id in relevant_ids else 0
        for chunk_id in retrieved_k
    ]

    # Precision@K
    precision = sum(relevance) / k

    # Recall@K
    recall = (
        sum(relevance) / len(relevant_ids)
        if relevant_ids else 0
    )

    # Reciprocal Rank
    reciprocal_rank = 0
    for rank, rel in enumerate(relevance, start=1):
        if rel:
            reciprocal_rank = 1 / rank
            break

    # NDCG@K (binary relevance)
    dcg = sum(
        rel / np.log2(rank + 1)
        for rank, rel in enumerate(relevance, start=1)
    )

    ideal_relevance = [1] * min(len(relevant_ids), k)

    idcg = sum(
        rel / np.log2(rank + 1)
        for rank, rel in enumerate(ideal_relevance, start=1)
    )

    ndcg = dcg / idcg if idcg else 0

    return precision, recall, reciprocal_rank, ndcg


retrieval_results = []

for item in evaluation_queries:

    retrieved = hybrid_retrieve_and_rerank(
        query=item["query"],
        candidate_k=20,
        final_top_k=20
    )

    retrieved_ids = [result["chunk_id"] for result in retrieved]

    for k in [5, 10, 20]:
        precision, recall, mrr, ndcg = calculate_retrieval_metrics(
            retrieved_ids,
            item["relevant_chunk_ids"],
            k
        )

        retrieval_results.append({
            "Query": item["query"],
            "Query Family": item["query_family"],
            "K": k,
            "Precision@K": precision,
            "Recall@K": recall,
            "MRR": mrr,
            "NDCG@K": ndcg
        })

retrieval_metrics_df = pd.DataFrame(retrieval_results)

display(retrieval_metrics_df.round(4))

metric_columns = ["Precision@K", "Recall@K", "MRR", "NDCG@K"]
print("Macro-average retrieval metrics by query family and K:")
display(
    retrieval_metrics_df.groupby(["Query Family", "K"])[metric_columns]
    .mean()
    .round(4)
)

,Query,Query Family,K,Precision@K,Recall@K,MRR,NDCG@K
0,What is the AISHE code of Regional Capacity Bu...,entity_lookup,5,0.20,1.0,1.0,1.0000
1,What is the AISHE code of Regional Capacity Bu...,entity_lookup,10,0.10,1.0,1.0,1.0000
2,What is the AISHE code of Regional Capacity Bu...,entity_lookup,20,0.05,1.0,1.0,1.0000
3,What is the AISHE code of THE NEOTIA UNIVERSITY?,entity_lookup,5,0.20,1.0,1.0,1.0000
4,What is the AISHE code of THE NEOTIA UNIVERSITY?,entity_lookup,10,0.10,1.0,1.0,1.0000
...,...,...,...,...,...,...,...
283,Which vidya lakshmi in Jharkhand were establis...,category_state_year,10,0.10,1.0,1.0,1.0000
284,Which vidya lakshmi in Jharkhand were establis...,category_state_year,20,0.05,1.0,1.0,1.0000
285,Which standalone in Jammu and Kashmir were est...,category_state_year,5,1.00,0.5,1.0,1.0000
286,Which standalone in Jammu and Kashmir were est...,category_state_year,10,0.50,0.5,1.0,0.6489


Macro-average retrieval metrics by query family and K:


Precision@K  Recall@K     MRR  NDCG@K
Query Family              K                                        
category_state            5        0.6583    0.6859  0.8750  0.8577
                          10       0.5000    0.8879  0.8819  0.8838
                          20       0.2604    0.9318  0.8849  0.8661
category_state_management 5        0.6833    0.5287  0.8576  0.7843
                          10       0.5208    0.7085  0.8636  0.7807
                          20       0.2792    0.7404  0.8636  0.7466
category_state_year       5        0.5250    0.9375  0.9792  0.9594
                          10       0.2750    0.9653  0.9792  0.9513
                          20       0.1375    0.9653  0.9792  0.9513
entity_lookup             5        0.2250    0.9583  0.9583  0.9583
                          10       0.1125    0.9583  0.9583  0.9583
                          20       0.0562    0.9583  0.9583  0.9583

In [24]:
for item in diagnostic_queries:

    print("\n" + "=" * 100)
    print("QUERY:", item["query"])

    results = hybrid_retrieve_and_rerank(
        query=item["query"],
        candidate_k=20,
        final_top_k=20
    )

    relevant_ids = set(item["relevant_chunk_ids"])

    for rank, result in enumerate(results, start=1):
        metadata = result["metadata"]

        print(
            f"{rank:02d}. "
            f"{'RELEVANT' if result['chunk_id'] in relevant_ids else 'NOT LABELLED'} | "
            f"{metadata['institution_name']} | "
            f"{metadata['aishe_code']} | "
            f"{metadata['institution_category']}"
        )


QUERY: List standalone in Chandigarh.
01. RELEVANT | GOVERNMENT MULTI SPECIALITY HOSPITAL CHANDIGARH | S-26921 | Standalone
02. RELEVANT | INDO SWISS TRAINING CENTRE | S-18386 | Standalone
03. RELEVANT | RAJARAM MOHAN RAI INSTITUTE OF VOCATIONAL STUDIES CHANDIGARH | S-5726 | Standalone
04. RELEVANT | CHANDIGARH INSTITUTE OF HOTEL MANAGEMENT AND CATERING TECHNOLOGY | S-18343 | Standalone
05. RELEVANT | Chandigarh College of Engineering and Technology (Diploma Wing) | S-15014 | Standalone
06. RELEVANT | D.A.V. MODEL SCHOOL CHANDIGARH | S-5725 | Standalone
07. RELEVANT | REGIONAL INSTITUTE OF COOPERATIVE MANAGEMENT | S-41 | Standalone
08. RELEVANT | GOVERNMENT POLYTECHNIC FOR WOMEN, CHANDIGARH | S-12472 | Standalone
09. NOT LABELLED | KANTI DARSHAN MAHAVIDYALAYA | S-26756 | Standalone
10. NOT LABELLED | SHRI KASHI CHANDRA DEV YADAV PRAVIDHIK SHIKSHAN SANSTHAN | S-3386 | Standalone
11. NOT LABELLED | KANTI DARSHAN MAHAVIDYALAYA | S-20359 | Standalone
12. NOT LABELLED | DIET DHARAMJAIGARH,

In [25]:
for item in diagnostic_queries:

    print("\n" + "=" * 100)
    print("QUERY:", item["query"])

    results = hybrid_retrieve_and_rerank(
        query=item["query"],
        candidate_k=20,
        final_top_k=20
    )

    for rank, result in enumerate(results, start=1):
        m = result["metadata"]

        print(
            f"{rank:02d}. {m['institution_name']}\n"
            f"    ID: {result['chunk_id']} | "
            f"AISHE: {m['aishe_code']}\n"
            f"    State: {m['state']} | "
            f"Category: {m['institution_category']}\n"
            f"    Management: {m['management_type']}\n"
        )


QUERY: List standalone in Chandigarh.
01. GOVERNMENT MULTI SPECIALITY HOSPITAL CHANDIGARH
    ID: INST-061126 | AISHE: S-26921
    State: Chandigarh | Category: Standalone
    Management: State Government

02. INDO SWISS TRAINING CENTRE
    ID: INST-062749 | AISHE: S-18386
    State: Chandigarh | Category: Standalone
    Management: Central Government

03. RAJARAM MOHAN RAI INSTITUTE OF VOCATIONAL STUDIES CHANDIGARH
    ID: INST-067284 | AISHE: S-5726
    State: Chandigarh | Category: Standalone
    Management: Private Un-Aided

04. CHANDIGARH INSTITUTE OF HOTEL MANAGEMENT AND CATERING TECHNOLOGY
    ID: INST-058810 | AISHE: S-18343
    State: Chandigarh | Category: Standalone
    Management: State Government

05. Chandigarh College of Engineering and Technology (Diploma Wing)
    ID: INST-058809 | AISHE: S-15014
    State: Chandigarh | Category: Standalone
    Management: Central Government

06. D.A.V. MODEL SCHOOL CHANDIGARH
    ID: INST-059233 | AISHE: S-5725
    State: Chandigarh 

In [26]:
import time
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

answer_eval_prompt = ChatPromptTemplate.from_template("""
You are an evaluator for a Retrieval-Augmented Generation system.

Evaluate the generated answer using the question and retrieved context.

QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{answer}

Evaluate these metrics on a scale of 1 to 5:

1. Faithfulness: Are the answer's claims supported by the retrieved context?
2. Answer Relevance: Does the answer directly address the question?
3. Citation Accuracy: Do the cited source references correspond to the supporting retrieved records?

Return ONLY in this format:

Faithfulness: <score>
Answer Relevance: <score>
Citation Accuracy: <score>
Reason: <brief explanation>
""")

answer_eval_chain = answer_eval_prompt | llm | StrOutputParser()

answer_quality_results = []

for item in llm_eval_queries:
    start_time = time.perf_counter()

    response = rag_chain.invoke(item["query"])

    answer = response["answer"]
    docs = response["docs"]

    context = "\n\n".join(
        f"[Source {doc.metadata['source_num']}] {doc.page_content}"
        for doc in docs
    )

    evaluation = answer_eval_chain.invoke({
        "question": item["query"],
        "context": context,
        "answer": answer
    })

    answer_quality_results.append({
        "Question": item["query"],
        "Answer": answer,
        "Evaluation": evaluation
    })

    print("=" * 100)
    print("QUESTION:", item["query"])
    print("ANSWER:", answer)
    print("\nEVALUATION:\n", evaluation)

answer_quality_df = pd.DataFrame(answer_quality_results)

QUESTION: List research in Nagaland.
ANSWER: Based on the provided AISHE records, there are two institutions engaged in research activities in Nagaland:

1. **ICAR - National Research Centre on Mithun, Nagaland**
   - **AISHE Code:** R-10082
   - **Category:** Research
   - **State:** Nagaland
   - **District:** Chumoukedima
   - **Administrative Ministry:** D/o Agricultural Research and Education, M/o Agriculture & Farmers Welfare

2. **Nagaland Science and Technology Council**
   - **AISHE Code:** R-10263
   - **Category:** Research
   - **State:** Nagaland
   - **District:** Kohima
   - **Administrative Ministry:** Ministry of Science & Technology

These institutions are dedicated to research activities in their respective fields.

**Sources:**
- ICAR - National Research Centre on Mithun, Nagaland | AISHE Code: R-10082 [Source 1]
- Nagaland Science and Technology Council | AISHE Code: R-10263 [Source 3]

EVALUATION:
 Faithfulness: 4
Answer Relevance: 5
Citation Accuracy: 5
Reason: T

In [27]:
import time
import numpy as np
import pandas as pd

performance_results = []

for item in performance_eval_queries:

    query = item["query"]

    # Retrieval and reranking latency
    start = time.perf_counter()

    results = hybrid_retrieve_and_rerank(
        query=query,
        candidate_k=20,
        final_top_k=5
    )

    retrieval_time = time.perf_counter() - start

    # End-to-end latency
    start = time.perf_counter()

    response = rag_chain.invoke(query)

    end_to_end_time = time.perf_counter() - start

    performance_results.append({
        "Query": query,
        "Retrieval + Reranking (ms)": retrieval_time * 1000,
        "End-to-End Latency (ms)": end_to_end_time * 1000,
        "Retrieved Documents": len(results)
    })

performance_df = pd.DataFrame(performance_results)

display(performance_df.round(2))

print("\nAverage Performance")
print("Average Retrieval + Reranking:",
      round(performance_df["Retrieval + Reranking (ms)"].mean(), 2), "ms")

print("Average End-to-End Latency:",
      round(performance_df["End-to-End Latency (ms)"].mean(), 2), "ms")

,Query,Retrieval + Reranking (ms),End-to-End Latency (ms),Retrieved Documents
0,List research in Bihar.,421.53,12498.68,5
1,List vidya lakshmi in Goa.,348.39,10927.22,5
2,List vidya lakshmi in Chandigarh.,502.41,15952.20,5
3,List research in Himachal Pradesh.,527.80,11703.85,5
4,List research in Meghalaya.,457.45,6887.29,5
5,Find Central Government standalone in Delhi.,435.12,8134.62,5
6,Find Central Government colleges in Bihar.,491.47,7670.66,5
7,Find Private Aided (Government Aided) colleges...,600.70,3953.27,5
8,Find Private Un-Aided standalone in Meghalaya.,490.35,7298.25,5
9,Find Local Body colleges in Chhattisgarh.,436.69,10783.59,5



Average Performance
Average Retrieval + Reranking: 618.39 ms
Average End-to-End Latency: 7904.27 ms


## Hybrid Retrieval

The CampusNLP system uses a hybrid retrieval approach combining dense and sparse retrieval to improve the relevance of institutional records.

**Retrieval Pipeline:**

1. **Dense Retrieval:** FAISS with `all-MiniLM-L6-v2` generates semantic embeddings and retrieves similar records.
2. **Sparse Retrieval:** BM25 retrieves records based on keyword matching.
3. **Reciprocal Rank Fusion (RRF):** Combines the rankings from dense and sparse retrieval.
4. **Cross-Encoder Reranking:** `cross-encoder/ms-marco-MiniLM-L-6-v2` reranks the retrieved candidates based on query-document relevance.

The final top-k records are passed to the RAG pipeline for answer generation.

## Generated Benchmark Results Analysis

This run evaluated **96 data-derived queries**: 24 each for entity lookup, category/state, category/state/management, and category/state/year. Every query has complete relevance labels from the AISHE fields and has at most 20 relevant records, so Recall@20 is meaningful.

### Overall verdict: good retrieval quality, with one important weakness

The retriever is **good for exact lookups and year-based filters**, and it is **good for category/state discovery** when using 10 results. It is weaker for management-type filtering, where roughly one quarter of relevant records are still missing even at K=20.

| Query family | Best practical K | Why | Assessment |
|---|---:|---|---|
| Category + state | **10** | Recall@10 = **0.8879**, NDCG@10 = **0.8838**, Precision@10 = 0.5000 | **Good** balance of coverage and relevance |
| Category + state + management | **10** | Precision@10 = 0.5208, Recall@10 = 0.7085 | **Moderate**; management constraints are the main retrieval gap |
| Category + state + year | **5** | Recall@5 = **0.9375**, MRR = **0.9792**, NDCG@5 = **0.9594** | **Excellent**; this is the strongest query family |
| Exact entity lookup | **5** | Recall@5 = **0.9583**, MRR = **0.9583**, NDCG@5 = **0.9583** | **Excellent**; the answer is normally found at rank 1 |

### What K should be used?

- **Use K=5** for direct AISHE-code questions and year-based filters. It already achieves very high recall and ranking quality while keeping the LLM context short.
- **Use K=10** as the default for category/state and management questions. It gives a much better recall/precision trade-off than K=5, while avoiding the less-relevant context introduced at K=20.
- **Use K=20** only when retrieval completeness matters more than concise context. For category/state, Recall improves from 0.8879 at K=10 to 0.9318 at K=20, but Precision drops from 0.5000 to 0.2604.

A falling Precision@K as K grows is expected: the system returns more documents while the number of relevant records stays fixed. For one-record entity queries, Precision@K should not be read as a failure; MRR, Recall, and NDCG are the useful measures.

### RAG answer quality and speed

On the 12-query LLM evaluation sample, average Faithfulness was **4.17/5**, Answer Relevance was **4.92/5**, and Citation Accuracy was **4.58/5**. This is good, but answer completeness can still suffer when the top-5 context omits valid records. Average retrieval plus reranking latency was **618 ms** and end-to-end latency was **7.90 s** across 20 queries.

### Recommended next improvement

For filters such as state, category, management type, and establishment year, apply an exact metadata filter before or alongside semantic retrieval. That will address the management-query ceiling (Recall@20 = 0.7404) and make broad listing questions more complete. Keep hybrid retrieval and cross-encoder reranking for entity and free-text questions.